In [1]:
# ============================================================
# 1. IMPORTACIONES Y CONFIGURACIÓN
# ============================================================

import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Buscar la raíz del proyecto para que el notebook funcione
# aunque Jupyter inicie desde otra carpeta.
CURRENT_DIR = Path.cwd().resolve()
ROOT_DIR = None
for directory in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (directory / "config.py").is_file():
        ROOT_DIR = directory
        break

if ROOT_DIR is None:
    raise FileNotFoundError("No se encontró config.py en el proyecto.")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, RANDOM_STATE

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. CARGAR DATASET ORIGINAL
# ============================================================

file_path = RAW_DATA_DIR / "breast_cancer_wisconsin.csv"
if not file_path.is_file():
    raise FileNotFoundError(f"No se encontró el dataset: {file_path}")

df = pd.read_csv(file_path)
print("Dataset cargado correctamente:", df.shape)


Dataset cargado correctamente: (569, 32)


In [2]:

# ============================================================
# 4. SEPARAR VARIABLES PREDICTORAS Y OBJETIVO
# ============================================================

# X contiene las 30 características predictoras
# Eliminamos ID porque es únicamente un identificador
X = df.drop(columns=["ID", "Diagnosis"])

# y contiene el diagnóstico que queremos predecir
# Benigno = 0
# Maligno = 1
y = df["Diagnosis"].map({
    "B": 0,
    "M": 1
})

# Verificar que la conversión fue correcta
if y.isna().any():
    raise ValueError("Se encontraron diagnósticos no reconocidos")

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nDistribución de las clases:")
print(y.value_counts().sort_index())


Dimensiones de X: (569, 30)
Dimensiones de y: (569,)

Distribución de las clases:
Diagnosis
0    357
1    212
Name: count, dtype: int64


In [3]:
# ============================================================
# 5. DIVISIÓN DE ENTRENAMIENTO, VALIDACIÓN Y PRUEBA
# ============================================================

# Respaldar las particiones anteriores para garantizar que la
# evaluación ya realizada siga utilizando los mismos registros.
ruta_x_test = PROCESSED_DATA_DIR / "X_test.csv"
ruta_y_test = PROCESSED_DATA_DIR / "y_test.csv"
ruta_metadata_test = PROCESSED_DATA_DIR / "test_metadata.csv"

X_test_anterior = pd.read_csv(ruta_x_test) if ruta_x_test.exists() else None
y_test_anterior = (
    pd.read_csv(ruta_y_test).iloc[:, 0]
    if ruta_y_test.exists() else None
)
metadata_test_anterior = (
    pd.read_csv(ruta_metadata_test)
    if ruta_metadata_test.exists() else None
)

# X contiene únicamente características predictoras; ID y Diagnosis
# se conservan aparte como metadatos.
metadata = df[["ID", "Diagnosis"]].copy()
if metadata["ID"].duplicated().any():
    raise ValueError("Existen identificadores duplicados en el dataset.")
if "ID" in X.columns or "Diagnosis" in X.columns:
    raise ValueError("ID y Diagnosis no deben formar parte de X.")

# Primera división: 70% entrenamiento y 30% temporal.
(
    X_train, X_temp,
    y_train, y_temp,
    metadata_train, metadata_temp
) = train_test_split(
    X, y, metadata,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Segunda división: 15% validación y 15% prueba.
(
    X_val, X_test,
    y_val, y_test,
    metadata_val, metadata_test
) = train_test_split(
    X_temp, y_temp, metadata_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# ============================================================
# 5.1. VERIFICAR QUE EL CONJUNTO DE PRUEBA NO CAMBIÓ
# ============================================================

if X_test_anterior is not None:
    if (
        X_test_anterior.shape != X_test.shape
        or list(X_test_anterior.columns) != list(X_test.columns)
        or not np.allclose(
            X_test_anterior.to_numpy(), X_test.to_numpy(),
            rtol=1e-10, atol=1e-12
        )
    ):
        raise ValueError("La partición X_test cambió inesperadamente.")

if y_test_anterior is not None:
    if not np.array_equal(
        pd.to_numeric(y_test_anterior).to_numpy(), y_test.to_numpy()
    ):
        raise ValueError("La partición y_test cambió inesperadamente.")

if metadata_test_anterior is not None:
    if not np.array_equal(
        metadata_test_anterior["ID"].to_numpy(), metadata_test["ID"].to_numpy()
    ):
        raise ValueError("Los identificadores de prueba cambiaron.")
    if not np.array_equal(
        metadata_test_anterior["Diagnosis"].to_numpy(), metadata_test["Diagnosis"].to_numpy()
    ):
        raise ValueError("Los diagnósticos de prueba cambiaron.")

# Verificar que los metadatos coincidan con las etiquetas en cada partición.
def verificar_metadata(y_particion, metadata_particion, nombre):
    diagnosticos = metadata_particion["Diagnosis"].map({"B": 0, "M": 1})
    if diagnosticos.isna().any():
        raise ValueError(f"Diagnósticos no reconocidos en {nombre}.")
    if not np.array_equal(diagnosticos.to_numpy(), y_particion.to_numpy()):
        raise ValueError(f"Los metadatos no coinciden con {nombre}.")
    if metadata_particion["ID"].duplicated().any():
        raise ValueError(f"Hay identificadores duplicados en {nombre}.")

verificar_metadata(y_train, metadata_train, "entrenamiento")
verificar_metadata(y_val, metadata_val, "validación")
verificar_metadata(y_test, metadata_test, "prueba")

# Comprobar que no haya registros compartidos entre particiones.
ids_train = set(metadata_train["ID"])
ids_val = set(metadata_val["ID"])
ids_test = set(metadata_test["ID"])
if ids_train & ids_val or ids_train & ids_test or ids_val & ids_test:
    raise ValueError("Existen identificadores compartidos entre particiones.")

# ============================================================
# 5.2. GUARDAR CARACTERÍSTICAS, ETIQUETAS Y METADATOS
# ============================================================

X_train.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_val.to_csv(PROCESSED_DATA_DIR / "X_val.csv", index=False)
X_test.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)

y_train.to_frame(name="Diagnosis").to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
y_val.to_frame(name="Diagnosis").to_csv(PROCESSED_DATA_DIR / "y_val.csv", index=False)
y_test.to_frame(name="Diagnosis").to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)

metadata_test = metadata_test.reset_index(drop=True)
metadata_test.to_csv(PROCESSED_DATA_DIR / "test_metadata.csv", index=False)

print("Particiones guardadas correctamente.")
print("Metadatos de prueba:", metadata_test.shape)
if X_test_anterior is not None:
    print("La partición anterior se conservó correctamente.")

# ============================================================
# 6. COMPROBAR LAS DIVISIONES
# ============================================================

print("\nDIMENSIONES DE LOS CONJUNTOS")
print("ENTRENAMIENTO:", X_train.shape, y_train.shape)
print("VALIDACIÓN:", X_val.shape, y_val.shape)
print("PRUEBA:", X_test.shape, y_test.shape)

print("\nDISTRIBUCIÓN DE CLASES")
nombres_clases = {0: "Benigno", 1: "Maligno"}
for nombre, target in [
    ("Entrenamiento", y_train),
    ("Validación", y_val),
    ("Prueba", y_test)
]:
    print(f"\n{nombre}:")
    distribucion = target.value_counts(normalize=True).sort_index().mul(100).round(2)
    for clase, porcentaje in distribucion.items():
        print(f"{nombres_clases[int(clase)]}: {porcentaje}%")

print("\nVERIFICACIÓN FINAL")
print("Total de registros:", len(X_train) + len(X_val) + len(X_test))
print("Identificadores únicos:", len(ids_train | ids_val | ids_test))
print("Variables predictoras:", X_train.shape[1])
print("Registros de entrenamiento:", len(X_train))
print("Registros de validación:", len(X_val))
print("Registros de prueba:", len(X_test))
print("\nTodas las comprobaciones finalizaron correctamente.")


Particiones guardadas correctamente.
Metadatos de prueba: (86, 2)
La partición anterior se conservó correctamente.

DIMENSIONES DE LOS CONJUNTOS
ENTRENAMIENTO: (398, 30) (398,)
VALIDACIÓN: (85, 30) (85,)
PRUEBA: (86, 30) (86,)

DISTRIBUCIÓN DE CLASES

Entrenamiento:
Benigno: 62.81%
Maligno: 37.19%

Validación:
Benigno: 62.35%
Maligno: 37.65%

Prueba:
Benigno: 62.79%
Maligno: 37.21%

VERIFICACIÓN FINAL
Total de registros: 569
Identificadores únicos: 569
Variables predictoras: 30
Registros de entrenamiento: 398
Registros de validación: 85
Registros de prueba: 86

Todas las comprobaciones finalizaron correctamente.


In [4]:

# ============================================================
# 7. ANALIZAR LAS ESCALAS DE LAS VARIABLES
# ============================================================

# Obtener estadísticas de las variables de entrenamiento
scale_summary = X_train.describe().T[
    ["mean", "std", "min", "max"]
]

# Ordenar por valor máximo
scale_summary = scale_summary.sort_values(
    by="max",
    ascending=False
)

display(scale_summary)


,mean,std,min,max
area3,883.117588,566.874169,185.200000,3432.00000
area1,655.325377,353.683472,143.500000,2499.00000
area2,40.296108,43.562155,6.802000,525.60000
perimeter3,107.401181,33.710958,50.410000,229.30000
perimeter1,91.904422,24.447698,43.790000,188.50000
texture3,25.920905,6.114636,12.020000,49.54000
texture1,19.438241,4.322444,9.710000,39.28000
radius3,16.296593,4.852525,7.930000,33.13000
radius1,14.126503,3.553198,6.981000,28.11000
perimeter2,2.867628,2.015020,0.757000,21.98000
